# Social Network Analysis of the Friends Graph

This notebook studies the character network contained in `data/friends_episodes.txt`. The raw file lists pairs of characters who interact during each episode and those interactions are aggregated here into a single graph in which nodes represent characters and edge weights count repeated co-occurrences across the series.

The text file contains 269 episode headers running from `s1e1` to `s10e18`. Throughout the notebook, this aggregated network is treated as undirected and weighted.

In [19]:
from __future__ import annotations

from collections import Counter
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from networkx.algorithms.community import (
    greedy_modularity_communities,
    label_propagation_communities,
)
from networkx.algorithms.community.quality import modularity

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:.6f}')
pio.templates.default = 'plotly_white'

DATA_DIR = Path('data')
OUTPUT_DIR = Path('out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DATA_DIR / 'friends_episodes.txt'
WEEK1_FIGURE_PATH = OUTPUT_DIR / 'week1_network.png'
WEEK3_FIGURE_PATH = OUTPUT_DIR / 'week3_clustering_ecdf.png'
WEEK6_NODES_PATH = OUTPUT_DIR / 'week6_nodes.csv'
WEEK6_EDGES_PATH = OUTPUT_DIR / 'week6_edges.csv'
PLOT_FONT = 'Georgia, Times New Roman, serif'
PLOT_BACKGROUND = '#ffffff'
RANDOM_SEED = 42


def parse_friends_dataset(path: Path) -> tuple[list[tuple[str, str, str]], list[str]]:
    rows: list[tuple[str, str, str]] = []
    episodes: list[str] = []
    current_episode = ''

    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('#'):
            current_episode = line[1:].strip()
            episodes.append(current_episode)
            continue

        left, right = line.split()
        rows.append((current_episode, left, right))

    return rows, episodes


def build_weighted_graph(rows: list[tuple[str, str, str]]) -> nx.Graph:
    edge_weights = Counter(tuple(sorted((left, right)))
                           for _, left, right in rows)
    graph = nx.Graph()
    graph.add_weighted_edges_from(
        (left, right, weight) for (left, right), weight in edge_weights.items()
    )
    return graph


def to_simple_graph(graph: nx.Graph) -> nx.Graph:
    return nx.Graph(graph)


def largest_component(graph: nx.Graph) -> nx.Graph:
    component_nodes = max(nx.connected_components(graph), key=len)
    return graph.subgraph(component_nodes).copy()


def basic_graph_stats(graph: nx.Graph) -> dict[str, float]:
    nodes = graph.number_of_nodes()
    edges = graph.number_of_edges()
    return {
        'nodes': float(nodes),
        'edges': float(edges),
        'average_degree': (2 * edges / nodes) if nodes else 0.0,
        'density': nx.density(graph),
        'connected_components': float(nx.number_connected_components(graph)),
    }


def rank_series(
    series: dict[str, float] | pd.Series,
    top_n: int = 10,
    column_name: str = 'score',
) -> pd.DataFrame:
    table = (
        pd.Series(series, name=column_name)
        .sort_values(ascending=False)
        .head(top_n)
        .rename_axis('character')
        .reset_index()
    )
    table.index = pd.RangeIndex(1, len(table) + 1, name='rank')
    return table


def rank_lookup(table: pd.DataFrame, column_name: str) -> pd.Series:
    return pd.Series(
        np.arange(1, len(table) + 1, dtype=int),
        index=table['character'],
        name=column_name,
    )


def style_plotly_figure(
    fig: go.Figure,
    *,
    title: str,
    x_title: str = '',
    y_title: str = '',
    width: int = 1100,
    height: int = 700,
) -> go.Figure:
    fig.update_layout(
        title={'text': title, 'x': 0.02, 'xanchor': 'left'},
        width=width,
        height=height,
        font={'family': PLOT_FONT, 'size': 16, 'color': '#1f2937'},
        paper_bgcolor=PLOT_BACKGROUND,
        plot_bgcolor='#fbfcfe',
        margin={'l': 60, 'r': 40, 't': 90, 'b': 60},
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.02,
            'xanchor': 'left',
            'x': 0.02,
            'bgcolor': 'rgba(255, 255, 255, 0.92)',
        },
    )
    fig.update_xaxes(
        title=x_title,
        showgrid=True,
        gridcolor='rgba(148, 163, 184, 0.22)',
        zeroline=False,
    )
    fig.update_yaxes(
        title=y_title,
        showgrid=True,
        gridcolor='rgba(148, 163, 184, 0.22)',
        zeroline=False,
    )
    return fig


def export_plotly_figure(fig: go.Figure, path: Path) -> None:
    fig.write_image(path, scale=2)

## Week 1

The first step is to construct the network in Python, visualize it, and compute the basic quantities that describe its overall shape. The deliverables for this week are a graph drawing together with the number of nodes, the number of edges, the average degree, the density, and a short interpretation of what these values say about the structure of the network. This is achieved by parsing the episode file, aggregating all character interactions into one weighted graph, and then computing the requested descriptive statistics on that aggregate object.

In [20]:
rows, episodes = parse_friends_dataset(DATASET_PATH)
G_weighted = build_weighted_graph(rows)
G_simple = to_simple_graph(G_weighted)
week1_stats = basic_graph_stats(G_simple)

summary_table = pd.Series(
    {
        'Episode count': len(episodes),
        'First episode marker': episodes[0],
        'Last episode marker': episodes[-1],
        'Nodes': int(week1_stats['nodes']),
        'Edges': int(week1_stats['edges']),
        'Average degree': week1_stats['average_degree'],
        'Density': week1_stats['density'],
        'Connected components': int(week1_stats['connected_components']),
    },
    name='value',
).to_frame()
display(summary_table)

layout = nx.spring_layout(G_simple, seed=RANDOM_SEED, k=0.22, iterations=120)
node_order = list(G_simple.nodes())
weighted_degree = dict(G_weighted.degree(weight='weight'))
node_frame = pd.DataFrame(
    {
        'character': node_order,
        'x': [layout[node][0] for node in node_order],
        'y': [layout[node][1] for node in node_order],
        'degree': [G_simple.degree(node) for node in node_order],
        'weighted_degree': [weighted_degree[node] for node in node_order],
    }
)
node_frame['size'] = 10 + 2.6 * np.sqrt(node_frame['degree'])
node_frame['color_value'] = np.log1p(node_frame['weighted_degree'])
label_frame = node_frame.nlargest(5, 'weighted_degree').copy()
label_offsets = [(0, -50), (70, -15), (-75, -10), (0, 55), (65, 45)]

edge_x: list[float | None] = []
edge_y: list[float | None] = []
for left, right in G_simple.edges():
    x0, y0 = layout[left]
    x1, y1 = layout[right]
    edge_x.extend((x0, x1, None))
    edge_y.extend((y0, y1, None))

network_figure = go.Figure(
    [
        go.Scatter(
            x=edge_x,
            y=edge_y,
            mode='lines',
            line={'color': 'rgba(71, 85, 105, 0.18)', 'width': 1},
            hoverinfo='skip',
            showlegend=False,
        ),
        go.Scatter(
            x=node_frame['x'],
            y=node_frame['y'],
            mode='markers',
            showlegend=False,
            customdata=node_frame[['character', 'degree', 'weighted_degree']],
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'Degree: %{customdata[1]}<br>'
                'Weighted degree: %{customdata[2]}<extra></extra>'
            ),
            marker={
                'size': node_frame['size'],
                'color': node_frame['color_value'],
                'colorscale': 'YlOrRd',
                'showscale': True,
                'colorbar': {
                    'thickness': 16,
                    'len': 0.72,
                    'y': 0.52,
                },
                'line': {'color': 'rgba(15, 23, 42, 0.65)', 'width': 0.7},
                'opacity': 0.96,
            },
        ),
    ]
)
style_plotly_figure(
    network_figure,
    title=(
        'Friends character interaction network'
        '<br><sup>Node size encodes degree and color encodes log-weighted degree across all episodes</sup>'
    ),
    width=1120,
    height=880,
)
zoom = 0.5
network_figure.update_xaxes(
    visible=False,
    range=[zoom * node_frame['x'].min(), zoom * node_frame['x'].max()],
)
network_figure.update_yaxes(
    visible=False,
    scaleanchor='x',
    scaleratio=1,
    range=[zoom * node_frame['y'].min(), zoom * node_frame['y'].max()],
)
network_annotations = [
    {
        'text': 'Layout: Fruchterman-Reingold spring embedding',
        'xref': 'paper',
        'yref': 'paper',
        'x': 0.02,
        'y': -0.06,
        'showarrow': False,
        'font': {'size': 12, 'color': '#475569'},
    }
]
for (offset_x, offset_y), (_, row) in zip(label_offsets, label_frame.iterrows()):
    network_annotations.append(
        {
            'x': row['x'],
            'y': row['y'],
            'xref': 'x',
            'yref': 'y',
            'text': row['character'],
            'showarrow': True,
            'arrowhead': 2,
            'arrowsize': 1,
            'arrowwidth': 1.1,
            'arrowcolor': '#475569',
            'ax': offset_x,
            'ay': offset_y,
            'bgcolor': 'rgba(255, 255, 255, 0.88)',
            'bordercolor': '#cbd5e1',
            'borderwidth': 1,
            'font': {'size': 13, 'color': '#0f172a'},
        }
    )
network_figure.update_layout(annotations=network_annotations)
export_plotly_figure(network_figure, WEEK1_FIGURE_PATH)
display(network_figure)

,value
Episode count,269
First episode marker,s1e1
Last episode marker,s10e18 last
Nodes,747
Edges,1610
Average degree,4.310576
Density,0.005778
Connected components,1


Across 269 episodes, the graph contains 747 characters but only 1,610 undirected ties, so density is just 0.0058 and the average node is linked to about 4.31 others. The figure is therefore dominated by a compact nucleus formed by the six protagonists, while hundreds of secondary characters occupy the periphery with only a few contacts. Yet the graph is still fully connected, which means those occasional characters are not isolated subworlds: they are attached back to the same narrative center. Treating the data as undirected remains appropriate because the file records joint appearance in an interaction, not an action sent from one character to another.

## Week 2

The second stage turns from global description to local cohesion by examining clustering on the largest connected component of the graph. The deliverables for this week are a manual implementation of the clustering coefficient for every node, a manual implementation of the average clustering coefficient, and a comparison between those values and the corresponding NetworkX functions for average clustering and transitivity. This is achieved by counting links among the neighbors of each node directly, validating the implementation on a toy graph, and then applying the same logic to the empirical network.

In [21]:
def node_clustering_manual(graph: nx.Graph) -> dict[str, float]:
    clustering: dict[str, float] = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        degree = len(neighbors)
        if degree < 2:
            clustering[node] = 0.0
            continue

        triangles = sum(
            1 for left, right in combinations(neighbors, 2) if graph.has_edge(left, right)
        )
        clustering[node] = (2 * triangles) / (degree * (degree - 1))
    return clustering


def average_clustering_manual(
    graph: nx.Graph,
    clustering_map: dict[str, float] | None = None,
) -> float:
    values = clustering_map or node_clustering_manual(graph)
    return float(pd.Series(values, dtype=float).mean())


validation_graph = nx.Graph([('A', 'B'), ('B', 'C'), ('A', 'C'), ('C', 'D')])
validation_manual = node_clustering_manual(validation_graph)
validation_expected = {'A': 1.0, 'B': 1.0, 'C': 1 / 3, 'D': 0.0}
assert all(
    abs(validation_manual[node] - validation_expected[node]) < 1e-12
    for node in validation_expected
)

G_week2 = largest_component(G_simple)
manual_clustering = node_clustering_manual(G_week2)
manual_average_clustering = average_clustering_manual(
    G_week2, manual_clustering)
networkx_average_clustering = nx.average_clustering(G_week2)
networkx_transitivity = nx.transitivity(G_week2)

week2_summary = pd.Series(
    {
        'Manual average clustering': manual_average_clustering,
        'NetworkX average clustering': networkx_average_clustering,
        'NetworkX transitivity': networkx_transitivity,
        'Maximum node clustering': max(manual_clustering.values()),
        'Minimum node clustering': min(manual_clustering.values()),
    },
    name='value',
).to_frame()
display(week2_summary)

top_clustering_table = rank_series(
    manual_clustering, top_n=10, column_name='clustering')
display(top_clustering_table)

,value
Manual average clustering,0.500264
NetworkX average clustering,0.500264
NetworkX transitivity,0.033501
Maximum node clustering,1.000000
Minimum node clustering,0.000000


,character,clustering
rank,,
1,waiterInDrag,1.000000
2,Alexandra,1.000000
3,Gina,1.000000
4,Cookie,1.000000
5,MaryTherese,1.000000
6,Don,1.000000
7,intern,1.000000
8,doctor_8_14,1.000000
9,LeonardHayes,1.000000


The manual computation reproduces the NetworkX average clustering exactly at 0.500264, so the implementation is correct. The substantive result, however, is the sharp contrast between that value and the much smaller transitivity of 0.033501. This gap says that local closure is common only in a very specific sense: many low-degree side characters sit in tiny fully closed neighborhoods, which gives them clustering 1, while the major characters create a huge number of open triplets simply because they connect to so many otherwise unrelated people. The top of the ranking confirms this pattern, since the nodes with perfect clustering are all minor characters rather than the central cast.

## Week 3

After computing local clustering values, the next task is to study how those values are distributed across the network rather than summarizing them with a single mean. The deliverables for this week are the cumulative distribution of node clustering, the cumulative distribution of the average clustering of each node's neighbors, and a comparison between the two curves. This is achieved by transforming both measures into empirical cumulative distribution functions and then plotting them together so that the relationship between personal local cohesion and neighborhood cohesion becomes visually clear.

In [22]:
def neighbor_average_clustering(
    graph: nx.Graph,
    clustering_map: dict[str, float],
) -> dict[str, float]:
    averages: dict[str, float] = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        averages[node] = float(np.mean([clustering_map[neighbor]
                               for neighbor in neighbors])) if neighbors else 0.0
    return averages


def ecdf(values: list[float] | pd.Series) -> pd.DataFrame:
    series = pd.Series(values, dtype=float).sort_values(ignore_index=True)
    return pd.DataFrame(
        {
            'value': series,
            'probability': (np.arange(len(series), dtype=float) + 1) / len(series),
        }
    )


neighbor_clustering = neighbor_average_clustering(G_week2, manual_clustering)
clustering_ecdf = ecdf(list(manual_clustering.values()))
neighbor_clustering_ecdf = ecdf(list(neighbor_clustering.values()))

plot_specs = [
    ('Node clustering', clustering_ecdf, '#b45309'),
    ('Average clustering of neighbors', neighbor_clustering_ecdf, '#1d4ed8'),
]
ecdf_figure = go.Figure()
for name, frame, color in plot_specs:
    ecdf_figure.add_trace(
        go.Scatter(
            x=frame['value'],
            y=frame['probability'],
            mode='lines',
            name=name,
            line={'width': 3, 'color': color, 'shape': 'hv'},
        )
    )

style_plotly_figure(
    ecdf_figure,
    title='Empirical cumulative distributions of clustering measures',
    x_title='Clustering value',
    y_title='Cumulative probability',
    width=1040,
    height=620,
)
ecdf_figure.update_yaxes(range=[0, 1.02], tickformat='.0%')
export_plotly_figure(ecdf_figure, WEEK3_FIGURE_PATH)
display(ecdf_figure)

week3_summary = pd.DataFrame(
    {
        'node_clustering': pd.Series(manual_clustering).describe(),
        'neighbor_average_clustering': pd.Series(neighbor_clustering).describe(),
    }
)
display(week3_summary)

,node_clustering,neighbor_average_clustering
count,747.000000,747.000000
mean,0.500264,0.122467
std,0.474916,0.170939
min,0.000000,0.000000
25%,0.000000,0.017855
50%,0.607143,0.021380
75%,1.000000,0.195799
max,1.000000,0.750000


The distributional picture is strongly polarized rather than smooth. About a quarter of the nodes have zero clustering, while the upper quartile is already at 1, so many characters are either embedded in completely open neighborhoods or in tiny perfectly closed ones. The ECDF makes the contrast with neighbor-average clustering even clearer: the neighborhood measure is concentrated near zero, with a median of only 0.021 and a mean of 0.122. In practice this means that a character can have high personal clustering because two or three of their contacts know each other, while still being attached to neighbors whose broader social surroundings are mostly open. The result is a network with many locally closed micro-scenes sitting inside a much less cohesive global environment.

## Week 5

The fifth week asks for a comparison of local notions of centrality, and this network lends itself naturally to two complementary measures. The deliverables for this week are two rankings of the most central characters together with an interpretation of who emerges as central under each definition. This is achieved by combining weighted PageRank, which rewards repeated and persistent interaction intensity, with betweenness centrality, which highlights characters who frequently sit on shortest paths and therefore connect different parts of the network.

In [23]:
weighted_pagerank = nx.pagerank(G_weighted, weight='weight')
betweenness = nx.betweenness_centrality(G_simple, weight=None)

pagerank_table = rank_series(
    weighted_pagerank, top_n=10, column_name='weighted_pagerank')
betweenness_table = rank_series(
    betweenness, top_n=10, column_name='betweenness')

for table in (pagerank_table, betweenness_table):
    display(table)

comparison_table = pd.concat(
    [
        rank_lookup(pagerank_table, 'pagerank_rank'),
        rank_lookup(betweenness_table, 'betweenness_rank'),
    ],
    axis=1,
).sort_values(['pagerank_rank', 'betweenness_rank'])
display(comparison_table)

,character,weighted_pagerank
rank,,
1,Chandler,0.125533
2,Joey,0.123319
3,Ross,0.122845
4,Monica,0.122291
5,Rachel,0.116383
6,Phoebe,0.109872
7,Mike,0.004762
8,Judy,0.004415
9,Jack,0.004259


,character,betweenness
rank,,
1,Joey,0.323798
2,Ross,0.271842
3,Rachel,0.238221
4,Chandler,0.231233
5,Phoebe,0.190730
6,Monica,0.183151
7,Mike,0.008299
8,Richard,0.006394
9,Joshua,0.005941


,pagerank_rank,betweenness_rank
character,,
Chandler,1.000000,4.000000
Joey,2.000000,1.000000
Ross,3.000000,2.000000
Monica,4.000000,6.000000
Rachel,5.000000,3.000000
Phoebe,6.000000,5.000000
Mike,7.000000,7.000000
Judy,8.000000,NaN
Jack,9.000000,NaN


The centrality rankings separate two forms of importance. Weighted PageRank is dominated by the six main characters, with Chandler, Joey, Ross, Monica, Rachel, and Phoebe occupying the first six positions because they accumulate repeated, heavy interaction volumes across the whole series. Betweenness keeps the same core cast near the top but reorders it, placing Joey first and Ross second, which indicates that they sit on more of the shortest paths linking otherwise separate parts of the graph. The contrast is most visible for characters such as Judy, Jack, and Gunther, who benefit from repeated links to the core and therefore appear in the PageRank top ten, but disappear from the betweenness top ten because they do not bridge the network. Conversely, Richard, Joshua, and Pete are stronger brokers than their raw interaction volume would suggest.

## Week 6

The sixth week shifts from ranking individual nodes to partitioning the network into communities. The deliverables for this week are the outputs of two community-detection methods, a modularity-based comparison between them, a justified choice of the better partition, and Gephi-ready node and edge files for visualization. This is achieved by removing any self-loops, converting the graph to the undirected and unweighted form required by the assignment, extracting the largest connected component, and then comparing greedy modularity optimization with label propagation on the same working graph.

In [24]:
def community_summary(
    graph: nx.Graph,
    communities: list[set[str]] | tuple[set[str], ...],
    method: str,
) -> dict[str, str | float | int]:
    community_sizes = sorted((len(community)
                             for community in communities), reverse=True)
    return {
        'method': method,
        'communities': len(communities),
        'modularity': modularity(graph, communities),
        'largest_sizes': ', '.join(map(str, community_sizes[:10])),
    }


G_week6 = largest_component(nx.Graph(G_simple))
G_week6.remove_edges_from(nx.selfloop_edges(G_week6))

partitions = {
    'Greedy modularity': [set(community) for community in greedy_modularity_communities(G_week6)],
    'Label propagation': [set(community) for community in label_propagation_communities(G_week6)],
}
week6_records = [
    community_summary(G_week6, partition, method)
    for method, partition in partitions.items()
]
week6_summary = pd.DataFrame(week6_records).sort_values(
    'modularity', ascending=False, ignore_index=True
)
display(week6_summary)

best_method = str(week6_summary.loc[0, 'method'])
best_partition = partitions[best_method]
community_lookup = {
    node: community_id
    for community_id, community in enumerate(best_partition, start=1)
    for node in sorted(community)
}

week6_nodes = pd.DataFrame(
    {
        'Id': list(G_week6.nodes()),
        'Label': list(G_week6.nodes()),
        'Community': [community_lookup[node] for node in G_week6.nodes()],
        'Degree': [G_week6.degree(node) for node in G_week6.nodes()],
        'WeightedPageRank': [weighted_pagerank[node] for node in G_week6.nodes()],
        'Betweenness': [betweenness[node] for node in G_week6.nodes()],
    }
).sort_values(['Community', 'Degree'], ascending=[True, False], ignore_index=True)

week6_edges = nx.to_pandas_edgelist(G_weighted).rename(
    columns={'source': 'Source', 'target': 'Target', 'weight': 'Weight'}
)
week6_nodes.to_csv(WEEK6_NODES_PATH, index=False)
week6_edges.to_csv(WEEK6_EDGES_PATH, index=False)

print(f'Selected partition: {best_method}')
print(f'Node export written to: {WEEK6_NODES_PATH}')
print(f'Edge export written to: {WEEK6_EDGES_PATH}')

display(week6_nodes.head(12))
display(week6_edges.head(12))

,method,communities,modularity,largest_sizes
0,Label propagation,21,0.005809,"694, 5, 4, 4, 3, 3, 3, 3, 3, 3"
1,Greedy modularity,11,-0.038086,"138, 126, 121, 120, 118, 109, 4, 4, 3, 2"


Selected partition: Label propagation
Node export written to: out/week6_nodes.csv
Edge export written to: out/week6_edges.csv


,Id,Label,Community,Degree,WeightedPageRank,Betweenness
0,Joey,Joey,1,268,0.123319,0.323798
1,Ross,Ross,1,243,0.122845,0.271842
2,Chandler,Chandler,1,237,0.125533,0.231233
3,Rachel,Rachel,1,228,0.116383,0.238221
4,Monica,Monica,1,207,0.122291,0.183151
5,Phoebe,Phoebe,1,206,0.109872,0.190730
6,Judy,Judy,1,19,0.004415,0.002641
7,Gunther,Gunther,1,18,0.002793,0.003323
8,Jack,Jack,1,17,0.004259,0.001621
9,Mike,Mike,1,17,0.004762,0.008299


,Source,Target,Weight
0,Joey,Monica,752
1,Joey,Chandler,1040
2,Joey,Ross,773
3,Joey,Rachel,711
4,Joey,Phoebe,713
5,Joey,Paul,2
6,Joey,Alan,1
7,Joey,nurseSizemore,1
8,Joey,Angela,6
9,Joey,Bob,2


The community-detection results are weak, and that weakness is itself informative. Label propagation performs better than greedy modularity in purely numerical terms, but its modularity is only 0.005809 and it assigns 694 of the 747 nodes to a single giant community, leaving only tiny residual groups. Greedy modularity is worse still, returning a negative modularity and therefore a partition that is not meaningful as evidence of real mesoscopic structure. The safest conclusion is that this graph is organized much more strongly around one dominant narrative core than around well-separated communities. If a partition must still be exported for Gephi Lite, label propagation is the defensible choice because it is the least bad option, but the visualization should be interpreted as highlighting a main component with marginal peripheral fragments, not a set of strong standalone communities.